# W03｜AsyncIO / 并发控制｜教学版

**建议学习时间：5h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射完成 AsyncIO / 并发控制 | FastAPI Ch9 + AsyncIO补充课指定小节 | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学并行 Tool 的适用边界 | 能判断哪些步骤可并行 | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 做 Sync vs Async 对比实验 | 记录 1/5/20 并发耗时 | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 并行执行 metadata + mock external call | 项目中至少 1 处真实 async | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 整理 Async / Thread / Process 对比 | 能按 IO/CPU 场景选型 | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 2｜并行边界参考答案
可并行：互不依赖的 metadata lookup、多个独立 HTTP/检索 Tool。  
不宜并行：B 依赖 A 输出、共享可变状态写入、必须保持顺序的副作用。

### Task 3｜1/5/20 验收
分别记录 sequential 与 concurrent 总耗时；并发 IO 应明显降低总等待时间，但不能因此无限并发。

### Task 4｜ChatBI 参考
```python
metadata, external = await asyncio.gather(
    get_metadata(),
    get_external_context()
)
```

### Task 5｜选型参考
Async → IO-bound；Thread → 同步阻塞 SDK/IO 兼容；Process → CPU-heavy。

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 理解 event loop/coroutine/task
- 能选择 Async vs Thread vs Process
- 会使用 Semaphore/Timeout 保护下游

        ## 本周前置检查

        - 完成 W01-W02；知道普通同步函数如何执行

        ## 本周必须记住的英文术语

        - `event loop`：事件循环
- `coroutine`：协程
- `TaskGroup`：任务组
- `Semaphore`：信号量
- `timeout`：超时
- `backpressure`：背压

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. AsyncIO 到底解决什么问题？

    **是什么（What）**  
    当程序大量时间在等待 IO（HTTP、LLM、DB）时，event loop 可以在等待期间切换去执行其他 coroutine。

    **为什么需要（Why）**  
    AI 应用经常要等 LLM API、MCP、DB，所以 async 能显著提高单进程的并发处理能力。

    **什么时候用（When）**  
    IO-bound workloads；不是重 CPU 计算。

    **常见误区 / 容易踩坑**  
    `async` 不等于“自动更快”；错误地调用阻塞函数会卡住整个 event loop。


**中英术语**：`event loop` = 事件循环；`coroutine` = 协程；`task` = 任务；`IO-bound` = IO密集；`CPU-bound` = CPU密集

### 代码实验：串行等待 vs 并发等待

**这段代码想让你看什么？**

两个版本做相同数量的 0.2 秒 IO；并发版总耗时应该明显更短。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
import asyncio, time

async def fake_io(i, delay=0.2):
    await asyncio.sleep(delay)
    return i

async def sequential():
    start = time.perf_counter()
    out = []
    for i in range(5):
        out.append(await fake_io(i))
    return time.perf_counter() - start

async def concurrent():
    start = time.perf_counter()
    out = await asyncio.gather(*(fake_io(i) for i in range(5)))
    return time.perf_counter() - start

print("sequential:", round(asyncio.run(sequential()), 3))
print("concurrent:", round(asyncio.run(concurrent()), 3))

## 2. 为什么还需要 Semaphore？

    **是什么（What）**  
    Semaphore 限制同一时间进入某段代码的任务数量。

    **为什么需要（Why）**  
    并发不是越大越好。LLM provider、数据库连接池、外部 API 都有容量上限。

    **什么时候用（When）**  
    保护下游、限流、避免瞬时 200 个请求把 provider 打爆。

    **常见误区 / 容易踩坑**  
    Semaphore 不是 retry，也不是 queue；它只是控制当前并发数。


**中英术语**：`semaphore` = 信号量/并发闸门；`backpressure` = 背压；`rate limit` = 限流

### 代码实验：验证 Semaphore 是否真的限制并发

**这段代码想让你看什么？**

`peak` 最终应该不超过 3。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
async def semaphore_demo():
    sem = asyncio.Semaphore(3)
    active = 0
    peak = 0
    lock = asyncio.Lock()

    async def guarded(i):
        nonlocal active, peak
        async with sem:
            async with lock:
                active += 1
                peak = max(peak, active)
            await asyncio.sleep(0.1)
            async with lock:
                active -= 1

    await asyncio.gather(*(guarded(i) for i in range(12)))
    return peak

print("peak concurrency:", asyncio.run(semaphore_demo()))

## 3. Timeout、Cancellation、to_thread

    **是什么（What）**  
    Timeout 限制等待时间；Cancellation 尝试取消未完成任务；`to_thread` 把阻塞函数放到线程，避免卡 event loop。

    **为什么需要（Why）**  
    Production API 不能无限等待下游；老的同步 SDK 也不能直接在 async endpoint 里长时间阻塞。

    **什么时候用（When）**  
    LLM timeout、HTTP tool timeout、兼容同步 SDK。

    **常见误区 / 容易踩坑**  
    Timeout 不代表远端一定没执行成功，这是 W09 idempotency 的重要前置。


**中英术语**：`timeout` = 超时；`cancellation` = 取消；`blocking call` = 阻塞调用

In [ ]:
async def timeout_demo():
    try:
        async with asyncio.timeout(0.1):
            await asyncio.sleep(1)
    except TimeoutError:
        return "timed out"

print(asyncio.run(timeout_demo()))

### 练习：并发实验

把任务数改成 5 / 20 / 100，把 Semaphore 改成 1 / 5 / 20，记录总耗时和 peak concurrency。

<details>
<summary><b>提示</b></summary>

用 `time.perf_counter()` 记录开始和结束时间。

</details>

In [ ]:
# TODO: 修改上面的 semaphore_demo，输出任务数、limit、耗时

<details>
    <summary><b>参考答案</b></summary>

    ```python
async def benchmark(n, limit):
    sem = asyncio.Semaphore(limit)
    start = time.perf_counter()
    async def one():
        async with sem:
            await asyncio.sleep(0.1)
    await asyncio.gather(*(one() for _ in range(n)))
    return time.perf_counter() - start

for n in [5,20,100]:
    for limit in [1,5,20]:
        print(n, limit, round(asyncio.run(benchmark(n, limit)), 3))
```

    </details>

### Checkpoint B：Thread / Process / Async

1. LLM API 调用更适合 Async 还是 Process？
2. 大型图片 CPU 推理预处理更适合 Async 还是 Process？
3. 同步第三方 SDK 在 async endpoint 里阻塞怎么办？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 通常 Async，因为主要等待网络 IO。

**2.** CPU-heavy 任务更适合 Process 或专门 worker；Async 不会提高 CPU 运算速度。

**3.** 可以换 async SDK，或短期用 `asyncio.to_thread()` 把阻塞调用移出 event loop。

    </details>

## ChatBI 本周作业

        在 ChatBI 里并行执行两个互不依赖的 IO：
- metadata lookup
- mock external call

加 `timeout + semaphore`。并解释为什么 DB pool 不能简单等于 HTTP 并发数。

### 本周面试 / 理解检查

1. Async 为什么适合 AI 应用？
2. 为什么不能无限 `gather` 1000 个 LLM 调用？
3. async def 中调用 `time.sleep(5)` 有什么问题？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 因为 LLM/DB/HTTP/MCP 多数是 IO-bound，等待期间 event loop 可处理其他任务。

**2.** 下游有并发/速率限制，可能导致 429、连接耗尽、内存增长，需要 semaphore/rate limit/backpressure。

**3.** 它会阻塞 event loop，导致同进程其他协程也无法及时执行。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。